In [1]:
import pandas as pd
import numpy as np
import joblib
import time

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [2]:
# =========================================================
# 1. CREATE TRANSACTION DATA
# =========================================================


In [3]:
data = {
    "Amount": [
        100, 200, 150, 5000, 300,
        10000, 250, 7000, 120, 8000,
        400, 15000, 180, 9000, 350,
        20000, 220, 6000, 450, 12000
    ],

    "Transactions_1H": [
        1, 2, 1, 10, 2,
        15, 1, 12, 1, 14,
        2, 18, 1, 13, 2,
        20, 2, 11, 3, 16
    ],

    "Previous_Fraud": [
        0, 0, 0, 1, 0,
        1, 0, 1, 0, 1,
        0, 1, 0, 1, 0,
        1, 0, 1, 0, 1
    ],

    "Distance_KM": [
        2, 5, 3, 500, 4,
        800, 6, 600, 3, 700,
        5, 900, 2, 650, 4,
        1000, 5, 550, 3, 750
    ],

    "Fraud": [
        0, 0, 0, 1, 0,
        1, 0, 1, 0, 1,
        0, 1, 0, 1, 0,
        1, 0, 1, 0, 1
    ]
}


In [4]:
df = pd.DataFrame(data)

print("DATASET")
print(df.head())
print("\nTotal Transactions:", len(df))
print("Fraud Transactions:", df["Fraud"].sum())

DATASET
   Amount  Transactions_1H  Previous_Fraud  Distance_KM  Fraud
0     100                1               0            2      0
1     200                2               0            5      0
2     150                1               0            3      0
3    5000               10               1          500      1
4     300                2               0            4      0

Total Transactions: 20
Fraud Transactions: 9


In [5]:
# =========================================================
# 2. FEATURES AND TARGET
# =========================================================


In [6]:
features = [
    "Amount",
    "Transactions_1H",
    "Previous_Fraud",
    "Distance_KM"
]

X = df[features]
y = df["Fraud"]


In [7]:
# =========================================================
# 3. TRAIN / TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=21,
    stratify=y
)



In [8]:
# =========================================================
# 4. RANDOM FOREST MODEL
# =========================================================

rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    class_weight="balanced",
    random_state=21
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("\n========== RANDOM FOREST ==========")
print("Accuracy :", round(accuracy_score(y_test, rf_pred), 3))
print("Precision:", round(precision_score(y_test, rf_pred), 3))
print("Recall   :", round(recall_score(y_test, rf_pred), 3))
print("F1 Score :", round(f1_score(y_test, rf_pred), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))



========== RANDOM FOREST ==========
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Confusion Matrix:
[[3 0]
 [0 2]]


In [9]:
# =========================================================
# 5. EXTRA TREES MODEL
# =========================================================

extra_model = ExtraTreesClassifier(
    n_estimators=150,
    max_depth=6,
    class_weight="balanced",
    random_state=21
)

extra_model.fit(X_train, y_train)

extra_pred = extra_model.predict(X_test)

print("\n========== EXTRA TREES ==========")
print("Accuracy :", round(accuracy_score(y_test, extra_pred), 3))
print("Precision:", round(precision_score(y_test, extra_pred), 3))
print("Recall   :", round(recall_score(y_test, extra_pred), 3))
print("F1 Score :", round(f1_score(y_test, extra_pred), 3))



========== EXTRA TREES ==========
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


In [10]:
# =========================================================
# 6. SAVE MODEL
# =========================================================

joblib.dump(rf_model, "fraud_detection_model.pkl")

print("\n✅ Fraud detection model saved as:")
print("fraud_detection_model.pkl")


# =========================================================
# 7. ANOMALY DETECTION
# =========================================================

anomaly_detector = IsolationForest(
    n_estimators=100,
    contamination=0.20,
    random_state=21
)

anomaly_detector.fit(X_train)

print("\n✅ Isolation Forest anomaly detector ready.")



✅ Fraud detection model saved as:
fraud_detection_model.pkl

✅ Isolation Forest anomaly detector ready.


In [11]:
# =========================================================
# 8. REAL-TIME FRAUD DETECTION FUNCTION
# =========================================================

def detect_transaction(transaction):

    new_transaction = pd.DataFrame(
        [transaction],
        columns=features
    )

    start_time = time.perf_counter()

    # Fraud prediction
    prediction = rf_model.predict(new_transaction)[0]

    # Fraud probability
    probability = (
        rf_model.predict_proba(new_transaction)[0][1]
        * 100
    )


In [16]:
def detect_transaction(transaction):

    new_transaction = pd.DataFrame(
        [transaction],
        columns=features
    )

    start_time = time.perf_counter()

    # Fraud prediction
    prediction = rf_model.predict(new_transaction)[0]

    # Fraud probability
    probability = (
        rf_model.predict_proba(new_transaction)[0][1]
        * 100
    )

    # Anomaly detection
    anomaly_result = anomaly_detector.predict(
        new_transaction
    )[0]

    latency = (
        time.perf_counter() - start_time
    ) * 1000

    # Risk Level
    if probability >= 75:
        risk = "HIGH"
    elif probability >= 40:
        risk = "MEDIUM"
    else:
        risk = "LOW"

    # Final Decision
    if prediction == 1:
        decision = "BLOCK / REVIEW"
    else:
        decision = "ALLOW"

    # Explanation
    reasons = []

    if transaction[0] > 5000:
        reasons.append("high transaction amount")

    if transaction[1] > 8:
        reasons.append("high transaction frequency")

    if transaction[2] == 1:
        reasons.append("previous fraud history")

    if transaction[3] > 300:
        reasons.append("unusual geographic distance")

    if len(reasons) == 0:
        explanation = "No major suspicious signal"
    else:
        explanation = ", ".join(reasons)

    anomaly_status = (
        "ANOMALOUS"
        if anomaly_result == -1
        else "NORMAL"
    )

    return {
        "Fraud Probability": round(probability, 2),
        "Risk Level": risk,
        "Decision": decision,
        "Anomaly": anomaly_status,
        "Explanation": explanation,
        "Latency (ms)": round(latency, 4)
    }

In [17]:
# =========================================================
# 9. STREAMING TRANSACTIONS
# =========================================================

live_transactions = [

    # Normal transaction
    [120, 2, 0, 5],

    # Suspicious transaction
    [9000, 14, 1, 700],

    # Normal transaction
    [250, 2, 0, 8],

    # Highly suspicious transaction
    [15000, 18, 1, 900],

    # Suspicious amount but lower frequency
    [6000, 8, 0, 500]
]


print("\n")
print("=" * 55)
print("        REAL-TIME FRAUD DETECTION")
print("=" * 55)


for number, transaction in enumerate(
    live_transactions,
    start=1
):

    result = detect_transaction(transaction)

    print(f"\nTransaction #{number}")
    print("-" * 40)

    print("Input:", transaction)

    print(
        "Fraud Probability:",
        result["Fraud Probability"],
        "%"
    )

    print(
        "Risk Level:",
        result["Risk Level"]
    )

    print(
        "Decision:",
        result["Decision"]
    )

    print(
        "Anomaly:",
        result["Anomaly"]
    )

    print(
        "Explanation:",
        result["Explanation"]
    )

    print(
        "Latency:",
        result["Latency (ms)"],
        "ms"
    )




        REAL-TIME FRAUD DETECTION

Transaction #1
----------------------------------------
Input: [120, 2, 0, 5]
Fraud Probability: 0.0 %
Risk Level: LOW
Decision: ALLOW
Anomaly: NORMAL
Explanation: No major suspicious signal
Latency: 191.6925 ms

Transaction #2
----------------------------------------
Input: [9000, 14, 1, 700]
Fraud Probability: 100.0 %
Risk Level: HIGH
Decision: BLOCK / REVIEW
Anomaly: NORMAL
Explanation: high transaction amount, high transaction frequency, previous fraud history, unusual geographic distance
Latency: 158.5687 ms

Transaction #3
----------------------------------------
Input: [250, 2, 0, 8]
Fraud Probability: 0.0 %
Risk Level: LOW
Decision: ALLOW
Anomaly: NORMAL
Explanation: No major suspicious signal
Latency: 110.191 ms

Transaction #4
----------------------------------------
Input: [15000, 18, 1, 900]
Fraud Probability: 100.0 %
Risk Level: HIGH
Decision: BLOCK / REVIEW
Anomaly: ANOMALOUS
Explanation: high transaction amount, high transaction frequ

In [19]:

# 10. FEATURE IMPORTANCE
# =========================================================

importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("\n========== FEATURE IMPORTANCE ==========")
print(importance)


# =========================================================
# 11. PROJECT SUMMARY
# =========================================================

print("\n")
print("=" * 55)
print("              PROJECT COMPLETED")
print("=" * 55)

print("✓ Fraud classification")
print("✓ Class imbalance handling")
print("✓ Random Forest model")
print("✓ Extra Trees comparison")
print("✓ Anomaly detection")
print("✓ Real-time transaction scoring")
print("✓ Risk classification")
print("✓ Fraud explanation")
print("✓ Inference latency measurement")
print("✓ Model saved successfully")



========== FEATURE IMPORTANCE ==========
           Feature  Importance
1  Transactions_1H    0.306667
0           Amount    0.273333
3      Distance_KM    0.240000
2   Previous_Fraud    0.180000


              PROJECT COMPLETED
✓ Fraud classification
✓ Class imbalance handling
✓ Random Forest model
✓ Extra Trees comparison
✓ Anomaly detection
✓ Real-time transaction scoring
✓ Risk classification
✓ Fraud explanation
✓ Inference latency measurement
✓ Model saved successfully
